# Travel Reimbursement Approval Agent

**Asrith Ladi** · AI Developer Candidate Assignment

This notebook takes an employee's travel expense claim, checks it against a written
reimbursement policy, and returns one of four recommendations: `APPROVE`, `PARTIAL_APPROVE`,
`REJECT` or `MANUAL_REVIEW`.

---

## README

### What this notebook does

It runs the five sample claims from Appendix B against the policy in Appendix A. Each claim comes
back as one JSON object: the decision, how much was approved, how much was deducted, any
paperwork that's missing, the policy rule ids behind the outcome, a confidence score, an
explanation in plain English, and the list of tools that actually ran.

### Setup steps

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

### Required environment variables

| Variable | Required? | Purpose |
|---|---|---|
| `OPENAI_API_KEY` | Recommended | Primary LLM provider. The agent's reasoning and explanations come from here. |
| `GROQ_API_KEY` | Optional | Fallback provider, used only if `OPENAI_API_KEY` is absent. |

Set it in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY=sk-...
```

A `.env` file in this folder works too. The key never gets written into the notebook, and
`.env` is gitignored.

### How to run the demo

1. `jupyter lab asrithladi.ipynb`
2. **Run All**. The cells are ordered so it goes top to bottom with nothing to do by hand.
3. Scroll to **## Dashboard** for the summary charts, and to the interactive UI cell if you want
   to try a claim of your own.
4. The last code cell prints the JSON array for all five claims.

### What happens if you don't have an API key

It won't crash. There are three modes, and the notebook prints which one it's in:

1. `OPENAI_API_KEY` set: full agentic run, LLM plans the tool calls and writes the explanations.
2. Only `GROQ_API_KEY` set: same agent, pointed at Groq's OpenAI-compatible endpoint.
3. Neither: the rule engine runs on its own. You get the same decisions and the same numbers,
   because the arithmetic never went through the LLM to begin with. The only thing that changes
   is the explanation text, from LLM-written to templated.

### The main design choices

**The LLM decides, Python calculates.** Every dollar figure in the output comes out of an
ordinary Python function. The model picks which checks to run and reads the results, but it
doesn't do the arithmetic. The numbers have to come out identical on every run.

**Policy comes from lookup, not from memory.** The policy is split one chunk per rule id and
retrieved on demand, so anything cited in `policy_refs` was genuinely read during that run.

**There's a guardrail after the agent.** It recalculates the numbers on its own, checks the
output against the schema, and pushes the claim to manual review if the two disagree.

**One agent, not a crew.** Five claims and twelve rules don't need several agents talking to each
other. A single LangGraph loop with decent tools covers it.

**When in doubt, ask a human.** A wrong approval pays out real money. An unnecessary manual
review costs a reviewer five minutes. That tells you which way to lean.

There's more on all of this, plus what I'd fix next, in the Design Notes near the end.

## 0. Setup and configuration

Everything tunable lives in this one cell: the policy limits, the approval tiers, the model name,
the safety thresholds. Nothing further down hardcodes a number. If finance raises the meal cap to
$85, that's a one-line change here and nothing else moves.

In [ ]:
import hashlib
import json
import os
import re
import textwrap
from datetime import date, datetime
from enum import Enum
from pathlib import Path
from typing import Any, Optional

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator

# .env is optional. Environment variables exported in the shell work the same way.
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

# LangChain sends traces to LangSmith when LANGSMITH_TRACING is set in the environment. Plenty of
# machines have it on from other work. Turn it off here so this notebook doesn't quietly ship claim
# data to a service nobody asked about, and doesn't fill the output with connection errors if that
# service is unreachable. Set it back to "true" yourself if you want the traces.
os.environ["LANGSMITH_TRACING"] = "false"

# --------------------------------------------------------------------------------------------
# LLM provider resolution: OpenAI first, Groq as fallback, deterministic mode if neither exists.
# --------------------------------------------------------------------------------------------
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "").strip()

if OPENAI_API_KEY:
    PROVIDER = "openai"
    MODEL = "gpt-4o-mini"  # swap to "gpt-4o" / "gpt-4.1" for the final run if you want
    API_KEY = OPENAI_API_KEY
    BASE_URL = None
elif GROQ_API_KEY:
    PROVIDER = "groq"
    MODEL = "llama-3.3-70b-versatile"
    API_KEY = GROQ_API_KEY
    BASE_URL = "https://api.groq.com/openai/v1"
else:
    PROVIDER = "none"
    MODEL = "(deterministic rule engine, no LLM)"
    API_KEY = None
    BASE_URL = None

LLM_AVAILABLE = PROVIDER != "none"

# --------------------------------------------------------------------------------------------
# Policy constants — Appendix A, expressed as data.
# --------------------------------------------------------------------------------------------
PER_DIEM_CAPS = {
    "meals": 75.0,             # POL-PD-01: per day
    "lodging": 200.0,          # POL-PD-02: per night
    "ground_transport": 50.0,  # POL-PD-03: per day
}

# Which unit each capped category is measured in. Lodging is billed per night; the rest per day.
CAP_UNIT = {"meals": "day", "lodging": "night", "ground_transport": "day"}

ELIGIBLE_CATEGORIES = {  # POL-CAT-01
    "airfare",
    "lodging",
    "meals",
    "ground_transport",
    "conference_fees",
}

INELIGIBLE_CATEGORIES = {  # POL-CAT-02
    "alcohol",
    "minibar",
    "spa",
    "gym",
    "entertainment",
    "in_room_movies",
    "personal_shopping",
    "gifts",
    "fines",
    "penalties",
    "late_fees",
    "personal",
}

# Free-text markers for ineligible spend, used when the category label alone is not conclusive.
INELIGIBLE_KEYWORDS = [
    "alcohol", "wine", "beer", "bar tab", "minibar", "mini-bar", "spa", "massage", "gym",
    "entertainment", "in-room movie", "movie rental", "personal shopping", "gift",
    "traffic fine", "penalty", "late fee",
]

# POL-AIR-01: premium cabin markers. Present means "policy exception", not "auto-deduct".
PREMIUM_CABIN_KEYWORDS = ["business class", "business-class", "first class", "first-class", "premium cabin"]

RECEIPT_THRESHOLD = 25.0                        # POL-RCT-01: receipt needed above this amount
ALWAYS_RECEIPT_CATEGORIES = {"airfare", "lodging"}  # POL-RCT-01: regardless of amount

TIER_AUTO_APPROVE = 500.0    # POL-APR-01: total <= 500
TIER_MANAGER = 2000.0        # POL-APR-02: 500 < total <= 2000; above this is POL-APR-03

SUBMISSION_WINDOW_DAYS = 30  # POL-TIME-01

# --------------------------------------------------------------------------------------------
# Agent safety limits.
# --------------------------------------------------------------------------------------------
MAX_AGENT_TURNS = 8      # hard cap on LLM<->tool round trips, so a confused model cannot loop
CONFIDENCE_FLOOR = 0.70  # below this, the decision is overridden to MANUAL_REVIEW
LLM_TEMPERATURE = 0.0    # we want the same answer every run

# --------------------------------------------------------------------------------------------
# Paths. Data files are written by the next cell if they do not already exist, so the notebook
# is self-contained even if someone downloads only the .ipynb.
# --------------------------------------------------------------------------------------------
DATA_DIR = Path("data")
POLICY_PATH = DATA_DIR / "policy.md"
CLAIMS_PATH = DATA_DIR / "claims.json"
CACHE_PATH = DATA_DIR / "llm_cache.json"
DATA_DIR.mkdir(exist_ok=True)

print("Travel Reimbursement Approval Agent")
print("-" * 64)
print(f"  LLM provider : {PROVIDER}")
print(f"  Model        : {MODEL}")
if not LLM_AVAILABLE:
    print("\n  No OPENAI_API_KEY or GROQ_API_KEY found.")
    print("  Running in DETERMINISTIC MODE: decisions and amounts are still produced by the")
    print("  policy tools; only the natural-language explanations are templated instead of")
    print("  LLM-written. Export a key and re-run to see the full agentic path.")
else:
    print(f"  Temperature  : {LLM_TEMPERATURE}  (deterministic)")
    print(f"  Max turns    : {MAX_AGENT_TURNS}")
print("-" * 64)

## 1. Inputs: the policy and the claims

Both inputs get written out to `data/` on the first run and then read back from disk. That looks
redundant, but it buys two things. The notebook still works if someone downloads only the
`.ipynb` and nothing else, and the claims genuinely arrive through a file read instead of sitting
as a Python literal in the middle of the decision code.

The policy uses one `###` heading per rule id. I formatted it that way on purpose so the chunker
can split on headings rather than guess at paragraph boundaries.

Claim descriptions are copied word for word from Appendix B, awkward ones like `"Hotel, 3 nights"`
included. Tidying those up would have quietly deleted the most interesting problem in the data.

In [ ]:
POLICY_MD = """\
# Travel Reimbursement Policy

Mock policy for this assignment. No real company or employee data. Every rule has a stable id
(POL-*) that must be cited in decision output where relevant.

## 1. Eligible & Ineligible Categories

### POL-CAT-01 — Eligible categories
Reimbursable when incurred for a documented business purpose:
- Airfare (economy class only — see POL-AIR-01)
- Lodging (hotel room charges)
- Meals (subject to per-diem limits — see POL-PD-01)
- Ground transport (taxi, rideshare, train, rental car, parking)
- Conference / registration fees

### POL-CAT-02 — Ineligible items
Never reimbursable; rejected and deducted in full:
- Alcohol and minibar charges
- Spa, gym, and personal entertainment
- In-room movies, personal shopping, gifts
- Traffic fines, penalties, and late fees
- Any personal (non-business) expense

## 2. Per-Diem & Category Limits

### POL-PD-01 — Meals per-diem
Maximum $75 per day. Amounts above the daily cap are deducted; the rest is reimbursed.

### POL-PD-02 — Lodging per-diem
Maximum $200 per night. Amounts above the nightly cap are deducted; the rest is reimbursed.

### POL-PD-03 — Ground transport per-diem
Maximum $50 per day. Amounts above the cap are deducted.

### POL-AIR-01 — Airfare class
Only economy class airfare is reimbursable. Business or first-class fares are a policy exception
and must be routed to Manual Review — they are not auto-deducted, because a pre-approval may exist.

## 3. Receipt Rules

### POL-RCT-01 — Receipt required above $25
Any single line item greater than $25 requires an attached, itemized receipt. Airfare and lodging
always require a receipt regardless of amount.

### POL-RCT-02 — Missing receipt handling
If a receipt is missing for an item that requires one, the item is not silently rejected — the
claim is routed to Manual Review so the reviewer can request the receipt.

## 4. Approval Thresholds

Thresholds are evaluated on the total reimbursable amount, after per-diem deductions but before
the final decision.

### POL-APR-01 — Auto-approve tier
Total less than or equal to $500 may be auto-approved by the agent if the claim is fully compliant.

### POL-APR-02 — Manager tier
Total greater than $500 and less than or equal to $2,000 is eligible for approval, and is treated
as approvable when the claim is fully compliant.

### POL-APR-03 — Director / Manual-Review tier
Total greater than $2,000 exceeds the agent's auto-approval authority and must be routed to Manual
Review because director approval is required, even if the claim is otherwise compliant.

## 5. Timeliness

### POL-TIME-01 — Submission window
Claims must be submitted within 30 days of the expense date. Late claims are routed to Manual
Review.
"""

# Decision guidance from Appendix A. This is instruction for the decider rather than a citable
# rule with an id, so it goes into the agent's system prompt instead of the retrievable chunks.
DECISION_GUIDANCE = """\
Approve — every item eligible, all receipts present, all within per-diem, total within an
approvable tier (POL-APR-01 / POL-APR-02).
Partially Approve — the claim is valid but some amounts exceed per-diem caps; reimburse up to the
cap and deduct the excess.
Reject — the claimed items are ineligible (POL-CAT-02) with nothing reimbursable.
Manual Review — any ambiguity, policy exception, high value (POL-APR-03), missing required receipt
(POL-RCT-02), or conflicting information. Prefer Manual Review over forcing a decision.
"""

# The five claims from Appendix B, transcribed exactly. Amounts in USD.
CLAIMS_RAW = [
    {
        "claim_id": "CLM-001",
        "employee": "A. Rivera",
        "purpose": "Attend 2-day industry conference (business)",
        "trip_start": "2026-06-10",
        "trip_end": "2026-06-12",
        "submitted_date": "2026-06-20",
        "currency": "USD",
        "line_items": [
            {"item_id": "CLM-001-L1", "category": "airfare", "description": "Round-trip economy airfare", "amount": 420.00, "receipt_attached": True},
            {"item_id": "CLM-001-L2", "category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": 360.00, "receipt_attached": True},
            {"item_id": "CLM-001-L3", "category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": 180.00, "receipt_attached": True},
            {"item_id": "CLM-001-L4", "category": "conference_fees", "description": "Conference registration", "amount": 150.00, "receipt_attached": True},
        ],
    },
    {
        "claim_id": "CLM-002",
        "employee": "B. Osei",
        "purpose": "Weekend hotel stay",
        "trip_start": "2026-06-14",
        "trip_end": "2026-06-15",
        "submitted_date": "2026-06-25",
        "currency": "USD",
        "line_items": [
            {"item_id": "CLM-002-L1", "category": "spa", "description": "Hotel spa package", "amount": 300.00, "receipt_attached": True},
            {"item_id": "CLM-002-L2", "category": "minibar", "description": "In-room minibar", "amount": 80.00, "receipt_attached": True},
        ],
    },
    {
        "claim_id": "CLM-003",
        "employee": "C. Nakamura",
        "purpose": "Client site visit (business)",
        "trip_start": "2026-06-08",
        "trip_end": "2026-06-10",
        "submitted_date": "2026-06-22",
        "currency": "USD",
        "line_items": [
            {"item_id": "CLM-003-L1", "category": "airfare", "description": "Round-trip economy airfare", "amount": 300.00, "receipt_attached": True},
            {"item_id": "CLM-003-L2", "category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": 500.00, "receipt_attached": True},
            {"item_id": "CLM-003-L3", "category": "meals", "description": "Meals, 2 days @ $70/day", "amount": 140.00, "receipt_attached": True},
        ],
    },
    {
        "claim_id": "CLM-004",
        "employee": "D. Fischer",
        "purpose": "International vendor negotiation (business)",
        "trip_start": "2026-06-16",
        "trip_end": "2026-06-18",
        "submitted_date": "2026-06-28",
        "currency": "USD",
        "line_items": [
            {"item_id": "CLM-004-L1", "category": "airfare", "description": "Business-class international airfare", "amount": 2400.00, "receipt_attached": True},
            {"item_id": "CLM-004-L2", "category": "lodging", "description": "Hotel, 3 nights", "amount": 600.00, "receipt_attached": False},
        ],
    },
    {
        "claim_id": "CLM-005",
        "employee": "E. Haddad",
        "purpose": "Client dinner / business development",
        "trip_start": "2026-06-11",
        "trip_end": "2026-06-11",
        "submitted_date": "2026-06-24",
        "currency": "USD",
        "line_items": [
            {"item_id": "CLM-005-L1", "category": "meals", "description": "Client dinner for 4 (business development)", "amount": 220.00, "receipt_attached": False},
        ],
    },
]

# Write once, then always read back from disk. That read is the notebook's real intake path.
if not POLICY_PATH.exists():
    POLICY_PATH.write_text(POLICY_MD)
if not CLAIMS_PATH.exists():
    CLAIMS_PATH.write_text(json.dumps(CLAIMS_RAW, indent=2))

policy_text = POLICY_PATH.read_text()
claims_from_disk = json.loads(CLAIMS_PATH.read_text())

print(f"Loaded policy : {POLICY_PATH}  ({len(policy_text):,} chars)")
print(f"Loaded claims : {CLAIMS_PATH}  ({len(claims_from_disk)} claims)")

## 2. Schemas: checking what goes in and what comes out

Two Pydantic models sit at the edges.

`Claim` checks the input. A malformed claim blows up here, at parse time, instead of turning into
a wrong dollar figure ten cells later. Setting `extra="forbid"` means a field I wasn't expecting
is an error rather than something silently ignored.

`ClaimResult` checks the output: the nine fields the assignment asks for, no more, with the
decision limited to the four allowed values. That's the contract the last cell prints.

`Claim` also works out the two numbers the per-diem rules need, nights and days, from the trip
dates. Separately, a small regex pulls any night or day count written into a line description.
Those two get compared later, and on one of the claims they don't agree.

In [ ]:
class Decision(str, Enum):
    """The only four decisions the system may emit."""

    APPROVE = "APPROVE"
    PARTIAL_APPROVE = "PARTIAL_APPROVE"
    REJECT = "REJECT"
    MANUAL_REVIEW = "MANUAL_REVIEW"


_UNIT_RE = re.compile(r"(\d+)\s*(night|day)s?\b", re.IGNORECASE)


class LineItem(BaseModel):
    """A single expense line on a claim."""

    model_config = ConfigDict(extra="forbid")

    item_id: str
    category: str
    description: str
    amount: float = Field(gt=0, description="Claimed amount in the claim's currency")
    receipt_attached: bool

    @property
    def stated_units(self) -> Optional[int]:
        """Unit count written in the description, e.g. 3 from 'Hotel, 3 nights'. None if absent.

        Deliberately does not match 'Client dinner for 4'. No unit word there, so the 4 is a
        headcount rather than a number of days.
        """
        match = _UNIT_RE.search(self.description)
        return int(match.group(1)) if match else None

    @property
    def stated_unit_kind(self) -> Optional[str]:
        """'night' or 'day' when the description states a unit, otherwise None."""
        match = _UNIT_RE.search(self.description)
        return match.group(2).lower() if match else None

    @property
    def needs_receipt(self) -> bool:
        """POL-RCT-01: over $25, or airfare/lodging at any amount."""
        return self.amount > RECEIPT_THRESHOLD or self.category in ALWAYS_RECEIPT_CATEGORIES


class Claim(BaseModel):
    """A complete reimbursement claim as submitted by an employee."""

    model_config = ConfigDict(extra="forbid")

    claim_id: str
    employee: str
    purpose: str
    trip_start: date
    trip_end: date
    submitted_date: date
    currency: str = "USD"
    line_items: list[LineItem] = Field(min_length=1)

    @model_validator(mode="after")
    def _check_dates(self) -> "Claim":
        if self.trip_end < self.trip_start:
            raise ValueError(f"{self.claim_id}: trip_end is before trip_start")
        if self.submitted_date < self.trip_start:
            raise ValueError(f"{self.claim_id}: submitted before the trip began")
        return self

    @property
    def nights(self) -> int:
        """Hotel nights implied by the trip dates: check-in to check-out."""
        return (self.trip_end - self.trip_start).days

    @property
    def days(self) -> int:
        """Calendar days the trip covers, inclusive of both endpoints. A same-day trip is 1 day."""
        return self.nights + 1

    @property
    def total_claimed(self) -> float:
        return round(sum(item.amount for item in self.line_items), 2)

    def units_for(self, category: str) -> int:
        """How many cap-units apply to a category, derived from the trip dates."""
        return self.nights if CAP_UNIT.get(category) == "night" else self.days


class ClaimResult(BaseModel):
    """The required output contract: nine fields, nothing more."""

    model_config = ConfigDict(extra="forbid")

    claim_id: str
    decision: Decision
    approved_amount: float = Field(ge=0)
    deducted_amount: float = Field(ge=0)
    missing_docs: list[str] = Field(default_factory=list)
    policy_refs: list[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    explanation: str
    tools_used: list[str] = Field(default_factory=list)

    @field_validator("approved_amount", "deducted_amount", "confidence")
    @classmethod
    def _round_numbers(cls, value: float) -> float:
        return round(float(value), 2)


# Parse and validate the claims we just read from disk.
claims: list[Claim] = [Claim.model_validate(raw) for raw in claims_from_disk]

# Intake sanity check against the totals printed in Appendix B. If a transcription slipped, this
# fails here rather than surfacing as a wrong decision much later.
EXPECTED_TOTALS = {
    "CLM-001": 1110.00,
    "CLM-002": 380.00,
    "CLM-003": 940.00,
    "CLM-004": 3000.00,
    "CLM-005": 220.00,
}
for claim in claims:
    expected = EXPECTED_TOTALS[claim.claim_id]
    assert claim.total_claimed == expected, (
        f"{claim.claim_id}: parsed total {claim.total_claimed} != Appendix B total {expected}"
    )

print(f"{len(claims)} claims validated against the Appendix B totals.\n")
header = f"{'claim':<9} {'employee':<13} {'trip':<25} {'nights':>6} {'days':>5} {'items':>6} {'total':>10}"
print(header)
print("-" * len(header))
for claim in claims:
    trip = f"{claim.trip_start} -> {claim.trip_end}"
    print(
        f"{claim.claim_id:<9} {claim.employee:<13} {trip:<25} {claim.nights:>6} "
        f"{claim.days:>5} {len(claim.line_items):>6} {claim.total_claimed:>10,.2f}"
    )

## 3. Grounding: looking the policy up instead of remembering it

The policy is about 2 KB. It would fit in the prompt with room to spare, so why retrieve it at
all?

Mainly so the citations mean something. If the agent has to call a lookup tool to see a rule,
every id in `policy_refs` came back from the actual document. Left to cite from memory, a model
will sooner or later produce a very confident `POL-PD-04`, which doesn't exist.

The other reason is that this mock policy stands in for a real one, and a real expense manual
runs to a couple of hundred pages. I'd rather not rewrite the agent when the corpus grows.

For the retrieval I used TF-IDF cosine similarity over the twelve rule chunks rather than
embeddings. No model download and no network call, which is what keeps this runnable on a machine
I've never seen. The downside is real: TF-IDF matches words, not meaning, so searching "hotel"
doesn't naturally find a chunk that says "lodging". I patched that specific hole with a small
alias map that expands the query before it goes in. It's a keyword system with a keyword fix, not
a semantic one. Swapping in embeddings later means changing one function.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Chunk on the '### POL-XXX' headings so we get one chunk per citable rule. The separator between
# the id and the title can be an em-dash, a hyphen or a colon, so that reformatting the policy
# document by hand cannot quietly break retrieval.
_CHUNK_RE = re.compile(r"^### (POL-[A-Z0-9\-]+)\s*[—–\-:]\s*(.+)$", re.MULTILINE)


def chunk_policy(markdown: str) -> list[dict[str, str]]:
    """Split the policy markdown into one chunk per POL-* rule id."""
    matches = list(_CHUNK_RE.finditer(markdown))
    chunks = []
    for index, match in enumerate(matches):
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown)
        body = markdown[start:end].strip()
        # Drop trailing '## Section' headings that belong to the next section, not this rule.
        body = re.split(r"^## ", body, maxsplit=1, flags=re.MULTILINE)[0].strip()
        chunks.append({"policy_id": match.group(1), "title": match.group(2).strip(), "text": body})
    return chunks


POLICY_CHUNKS = chunk_policy(policy_text)
VALID_POLICY_IDS = {chunk["policy_id"] for chunk in POLICY_CHUNKS}

# If the policy file ever gets reformatted in a way the chunker does not understand, fail here with
# something readable. Without this the next line dies inside scikit-learn on an empty vocabulary,
# which tells you nothing about the actual cause.
assert len(POLICY_CHUNKS) == 12, (
    "expected 12 policy rules, found " + str(len(POLICY_CHUNKS))
    + ". Check that every rule in data/policy.md still starts with '### POL-...'."
)

# Query expansion: map everyday words onto the vocabulary the policy actually uses. This is the
# cheap, explainable fix for TF-IDF's blindness to synonyms.
QUERY_ALIASES = {
    "hotel": "lodging nightly", "room": "lodging", "stay": "lodging", "night": "lodging nightly",
    "food": "meals per-diem", "dinner": "meals", "lunch": "meals", "breakfast": "meals",
    "flight": "airfare", "plane": "airfare", "fare": "airfare", "cabin": "airfare class economy",
    "taxi": "ground transport", "cab": "ground transport", "uber": "ground transport",
    "rideshare": "ground transport", "train": "ground transport", "parking": "ground transport",
    "rental": "ground transport",
    "bill": "receipt", "invoice": "receipt itemized", "proof": "receipt",
    "cap": "maximum per-diem limit", "limit": "maximum per-diem", "over": "above cap deducted",
    "late": "submission window 30 days", "deadline": "submission window", "delay": "submission window",
    "threshold": "approval tier total", "tier": "approval threshold total",
    "authority": "approval tier director",
    "business class": "airfare class economy exception", "upgrade": "airfare class exception",
    "spa": "ineligible", "minibar": "ineligible alcohol", "alcohol": "ineligible",
    "gym": "ineligible", "movie": "ineligible entertainment", "fine": "ineligible penalties",
    # Stems rather than whole words, so 'approves'/'approval'/'approve' all expand.
    "approv": "approval threshold tier total director",
    "eligib": "eligible categories reimbursable",
    "reimburs": "reimbursable eligible",
}


class PolicyRetriever:
    """Tiny TF-IDF retriever over the policy chunks. No network, no model download."""

    def __init__(self, chunks: list[dict[str, str]]) -> None:
        self.chunks = chunks
        self._corpus = [f"{c['policy_id']} {c['title']} {c['text']}" for c in chunks]
        self._vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), stop_words="english", sublinear_tf=True
        )
        self._matrix = self._vectorizer.fit_transform(self._corpus)

    def _expand(self, query: str) -> str:
        lowered = query.lower()
        extras = [value for key, value in QUERY_ALIASES.items() if key in lowered]
        return f"{query} {' '.join(extras)}".strip()

    def search(self, query: str, top_k: int = 3) -> list[dict[str, Any]]:
        """Return the top_k most relevant policy chunks, each with its similarity score.

        An exact POL-* id in the query short-circuits the search: if the caller already knows the
        rule it wants, give it to them rather than guessing.
        """
        explicit = [c for c in self.chunks if c["policy_id"] in query.upper()]
        if explicit:
            return [{**chunk, "score": 1.0, "matched_by": "explicit_id"} for chunk in explicit]

        vector = self._vectorizer.transform([self._expand(query)])
        scores = cosine_similarity(vector, self._matrix)[0]
        ranked = sorted(enumerate(scores), key=lambda pair: pair[1], reverse=True)[:top_k]
        return [
            {**self.chunks[index], "score": round(float(score), 4), "matched_by": "tfidf"}
            for index, score in ranked
            if score > 0
        ]


retriever = PolicyRetriever(POLICY_CHUNKS)

print(f"Policy chunked into {len(POLICY_CHUNKS)} citable rules:")
for chunk in POLICY_CHUNKS:
    print(f"  {chunk['policy_id']:<13} {chunk['title']}")

print("\nRetrieval smoke test. Everyday phrasing should still find the right rule:")
for probe in ["hotel is too expensive per night", "no bill attached for the flight",
              "claim submitted very late", "who approves a $3000 claim"]:
    hits = retriever.search(probe, top_k=2)
    found = ", ".join(f"{h['policy_id']} ({h['score']})" for h in hits)
    print(f"  {probe:<38} -> {found}")

## 4. The tools: deterministic policy checks

This is where the reimbursement logic actually lives, and the LLM isn't involved in any of it.

Here's the reasoning. Ask a language model "the hotel is $500 for 2 nights and the cap is $200 a
night, what do I deduct?" and it'll usually say $100. Usually. Every so often it says $300, and
it won't necessarily give you the same answer twice on the same input. That's fine for drafting
an email and not fine for deciding what somebody gets paid. So all the arithmetic below is plain
Python. The model's job is picking which checks to run, reading what comes back, sorting out
disagreements between them, and writing the explanation.

Seven tools, one per area of the policy:

| Tool | Answers | Rules |
|---|---|---|
| `policy_lookup` | what does the policy actually say about X? | all |
| `eligibility_check` | is this category reimbursable at all? | POL-CAT-01/02, POL-AIR-01 |
| `per_diem_check` | how much of this is over the caps? | POL-PD-01/02/03 |
| `receipt_check` | what paperwork is missing? | POL-RCT-01/02 |
| `timeliness_check` | was this submitted in time? | POL-TIME-01 |
| `threshold_check` | is this within the agent's authority? | POL-APR-01/02/03 |
| `duplicate_check` | have we already paid this? | cross-claim control |

Two things to know before reading the code.

Each tool returns its own `policy_refs`. The citation list on the final result gets built from
what the tools actually looked at, so the agent can't pin a decision on a rule nobody checked.

Tools return errors as data instead of raising. Ask about `CLM-999` and you get back
`{"error": "unknown claim_id ..."}`, which the model can read and correct. An exception would
just kill the run.

In [ ]:
# The tools look claims up by id from here. The interactive UI adds ad-hoc claims to this same
# dictionary, so a claim you type in by hand goes through exactly the same checks.
CLAIM_REGISTRY = {}
for claim in claims:
    CLAIM_REGISTRY[claim.claim_id] = claim


def find_claim(claim_id):
    """Look up a claim by id. Returns None if we don't have it."""
    return CLAIM_REGISTRY.get(claim_id)


def unknown_claim_error(tool_name, claim_id):
    """Build the error a tool returns when it's handed a claim id we don't have."""
    known_ids = ", ".join(sorted(CLAIM_REGISTRY))
    return {
        "tool": tool_name,
        "error": "unknown claim_id '" + claim_id + "'. Known ids: " + known_ids,
    }

In [ ]:
def policy_lookup(query, top_k=3):
    """Find the policy rules relevant to a plain-language question."""
    hits = retriever.search(query, top_k=top_k)

    refs = []
    for hit in hits:
        refs.append(hit["policy_id"])

    return {"tool": "policy_lookup", "query": query, "results": hits, "policy_refs": refs}


def is_premium_cabin(description):
    """True if an airfare description mentions a cabin above economy."""
    lowered = description.lower()
    for keyword in PREMIUM_CABIN_KEYWORDS:
        if keyword in lowered:
            return True
    return False


def find_ineligible_keyword(description):
    """Return the first banned word found in a description, or None."""
    lowered = description.lower()
    for keyword in INELIGIBLE_KEYWORDS:
        if keyword in lowered:
            return keyword
    return None


def eligibility_check(claim_id):
    """Sort every line into eligible, ineligible, or unrecognised (POL-CAT-01 and POL-CAT-02).

    One thing to watch here. Business-class airfare comes back as a policy *exception*, not as an
    ineligible item. POL-AIR-01 says send it to a human rather than deduct it, because there may
    be a pre-approval somewhere that this system can't see.
    """
    claim = find_claim(claim_id)
    if claim is None:
        return unknown_claim_error("eligibility_check", claim_id)

    checked_items = []
    exceptions = []
    refs = set()
    eligible_total = 0.0
    ineligible_total = 0.0
    unknown_total = 0.0

    for item in claim.line_items:
        banned_word = find_ineligible_keyword(item.description)

        if item.category in INELIGIBLE_CATEGORIES:
            status = "ineligible"
            reason = "category '" + item.category + "' is listed in POL-CAT-02"
            refs.add("POL-CAT-02")
            ineligible_total += item.amount

        elif banned_word is not None:
            status = "ineligible"
            reason = "description mentions '" + banned_word + "', which POL-CAT-02 excludes"
            refs.add("POL-CAT-02")
            ineligible_total += item.amount

        elif item.category in ELIGIBLE_CATEGORIES:
            status = "eligible"
            reason = "category '" + item.category + "' is listed in POL-CAT-01"
            refs.add("POL-CAT-01")
            eligible_total += item.amount

        else:
            # Not on either list. We don't guess in favour of the employee or the company.
            status = "unrecognised"
            reason = "category '" + item.category + "' is not named in POL-CAT-01 or POL-CAT-02"
            refs.add("POL-CAT-01")
            unknown_total += item.amount

        # POL-AIR-01: flag a premium cabin, but don't deduct it.
        if item.category == "airfare" and is_premium_cabin(item.description):
            exceptions.append({
                "item_id": item.item_id,
                "type": "premium_cabin_airfare",
                "amount": round(item.amount, 2),
                "policy_ref": "POL-AIR-01",
                "detail": "Non-economy airfare. POL-AIR-01 sends this to Manual Review instead of "
                          "deducting it, since a pre-approval may exist.",
            })
            refs.add("POL-AIR-01")

        checked_items.append({
            "item_id": item.item_id,
            "category": item.category,
            "amount": round(item.amount, 2),
            "status": status,
            "reason": reason,
        })

    nothing_reimbursable = ineligible_total > 0 and eligible_total == 0 and unknown_total == 0

    return {
        "tool": "eligibility_check",
        "claim_id": claim_id,
        "items": checked_items,
        "eligible_total": round(eligible_total, 2),
        "ineligible_total": round(ineligible_total, 2),
        "unrecognised_total": round(unknown_total, 2),
        "policy_exceptions": exceptions,
        "all_items_ineligible": nothing_reimbursable,
        "requires_manual_review": len(exceptions) > 0 or unknown_total > 0,
        "policy_refs": sorted(refs),
    }

In [ ]:
# Which rule to cite for each capped category.
CAP_POLICY_REF = {
    "meals": "POL-PD-01",
    "lodging": "POL-PD-02",
    "ground_transport": "POL-PD-03",
}


def per_diem_check(claim_id):
    """Apply the per-diem caps and work out the excess on each capped line.

    The fiddly part is deciding how many units to multiply the cap by. Lodging is capped per night
    and meals per day, so we need a count, and there are two places to get one: the trip dates in
    the claim header, or a number written into the description like "Hotel, 3 nights".

    I treat the trip dates as the real answer. When the description disagrees, the question is
    whether it changes the money. If both readings give the same deduction it doesn't matter, so
    we note it and carry on. If they give different deductions that's a genuine conflict, so we
    hold the deduction back and send the claim to a human. Picking one at random would mean
    guessing at somebody's money.
    """
    claim = find_claim(claim_id)
    if claim is None:
        return unknown_claim_error("per_diem_check", claim_id)

    lines = []
    conflicts = []
    refs = set()
    total_excess = 0.0

    for item in claim.line_items:
        cap = PER_DIEM_CAPS.get(item.category)

        # Airfare and conference fees have no cap, so the whole amount is allowed.
        if cap is None:
            lines.append({
                "item_id": item.item_id,
                "category": item.category,
                "amount": round(item.amount, 2),
                "capped": False,
                "allowed": round(item.amount, 2),
                "excess": 0.0,
                "note": "no per-diem cap applies to this category",
            })
            continue

        unit = CAP_UNIT[item.category]
        policy_ref = CAP_POLICY_REF[item.category]
        refs.add(policy_ref)

        # What the trip dates say.
        trip_units = claim.units_for(item.category)
        allowed = round(cap * trip_units, 2)
        excess = round(max(0.0, item.amount - allowed), 2)

        conflict = None

        # Does the description state a different count, in the same unit?
        description_disagrees = (
            item.stated_units is not None
            and item.stated_unit_kind == unit
            and item.stated_units != trip_units
        )

        if description_disagrees:
            allowed_if_description = round(cap * item.stated_units, 2)
            excess_if_description = round(max(0.0, item.amount - allowed_if_description), 2)

            # Only a problem if the two readings actually pay out differently.
            changes_the_money = abs(excess_if_description - excess) >= 0.01

            if changes_the_money:
                detail = (
                    "Trip dates (" + str(claim.trip_start) + " to " + str(claim.trip_end) + ") "
                    "imply " + str(trip_units) + " " + unit + "(s), but the line reads '"
                    + item.description + "'. The two readings deduct different amounts "
                    "($" + format(excess, ".2f") + " vs $" + format(excess_if_description, ".2f")
                    + "), so the deduction is held back for review."
                )
            else:
                detail = (
                    "Trip dates imply " + str(trip_units) + " " + unit + "(s) but the line says "
                    + str(item.stated_units) + ". Both readings deduct the same amount, so it "
                    "makes no difference here."
                )

            conflict = {
                "item_id": item.item_id,
                "type": "unit_count_conflict",
                "trip_dates_imply": str(trip_units) + " " + unit + "(s)",
                "description_states": str(item.stated_units) + " " + unit + "(s)",
                "excess_if_trip_dates": excess,
                "excess_if_description": excess_if_description,
                "material": changes_the_money,
                "policy_ref": policy_ref,
                "detail": detail,
            }

            if changes_the_money:
                conflicts.append(conflict)

        # A per-night item on a trip with no nights can't be checked at all.
        if unit == "night" and trip_units == 0:
            conflict = {
                "item_id": item.item_id,
                "type": "zero_nights",
                "material": True,
                "policy_ref": policy_ref,
                "detail": "Lodging claimed on a trip whose dates cover zero nights.",
            }
            conflicts.append(conflict)

        deduction_held_back = conflict is not None and conflict["material"]
        if deduction_held_back:
            reported_excess = 0.0
        else:
            reported_excess = excess
            total_excess += excess

        lines.append({
            "item_id": item.item_id,
            "category": item.category,
            "amount": round(item.amount, 2),
            "capped": True,
            "cap_per_unit": cap,
            "unit": unit,
            "units_used": trip_units,
            "allowed": allowed,
            "excess": reported_excess,
            "deduction_held_pending_review": deduction_held_back,
            "policy_ref": policy_ref,
            "conflict": conflict,
        })

    return {
        "tool": "per_diem_check",
        "claim_id": claim_id,
        "trip_nights": claim.nights,
        "trip_days": claim.days,
        "lines": lines,
        "total_excess": round(total_excess, 2),
        "conflicts": conflicts,
        "requires_manual_review": len(conflicts) > 0,
        "policy_refs": sorted(refs),
    }

In [ ]:
def receipt_check(claim_id):
    """List the receipts the policy requires but the claim doesn't have (POL-RCT-01, POL-RCT-02)."""
    claim = find_claim(claim_id)
    if claim is None:
        return unknown_claim_error("receipt_check", claim_id)

    checked_items = []
    missing = []
    refs = set()

    for item in claim.line_items:
        required = item.needs_receipt

        if required:
            refs.add("POL-RCT-01")
            if item.category in ALWAYS_RECEIPT_CATEGORIES:
                why = item.category + " always needs a receipt, whatever the amount"
            else:
                why = ("$" + format(item.amount, ",.2f") + " is over the $"
                       + format(RECEIPT_THRESHOLD, ",.0f") + " threshold")
        else:
            why = ("$" + format(item.amount, ",.2f") + " is at or under the $"
                   + format(RECEIPT_THRESHOLD, ",.0f") + " threshold")

        if required and not item.receipt_attached:
            missing.append(
                "Itemized receipt for " + item.category + ": " + item.description
                + " ($" + format(item.amount, ",.2f") + ")"
            )
            # POL-RCT-02 is explicit that this is a review trigger, not a rejection.
            refs.add("POL-RCT-02")
            satisfied = False
        else:
            satisfied = True

        checked_items.append({
            "item_id": item.item_id,
            "category": item.category,
            "amount": round(item.amount, 2),
            "receipt_required": required,
            "receipt_attached": item.receipt_attached,
            "requirement_reason": why,
            "satisfied": satisfied,
        })

    return {
        "tool": "receipt_check",
        "claim_id": claim_id,
        "items_checked": checked_items,
        "missing_docs": missing,
        "requires_manual_review": len(missing) > 0,
        "policy_refs": sorted(refs),
    }


def timeliness_check(claim_id):
    """Check the 30-day submission window (POL-TIME-01).

    The policy says "within 30 days of the expense date", but a claim covers a range of dates and
    the individual lines don't carry their own. I measure from trip_start, the earliest any expense
    on the claim could have happened, which is the strictest reading. If measuring from trip_end
    would flip the answer, that gets reported instead of quietly resolved.
    """
    claim = find_claim(claim_id)
    if claim is None:
        return unknown_claim_error("timeliness_check", claim_id)

    days_from_start = (claim.submitted_date - claim.trip_start).days
    days_from_end = (claim.submitted_date - claim.trip_end).days

    late_strict = days_from_start > SUBMISSION_WINDOW_DAYS
    late_lenient = days_from_end > SUBMISSION_WINDOW_DAYS
    depends_on_reading = late_strict != late_lenient

    return {
        "tool": "timeliness_check",
        "claim_id": claim_id,
        "submitted_date": str(claim.submitted_date),
        "window_days": SUBMISSION_WINDOW_DAYS,
        "days_since_trip_start": days_from_start,
        "days_since_trip_end": days_from_end,
        "is_late": late_strict,
        "basis": "trip_start (strictest reading)",
        "ambiguous_window": depends_on_reading,
        "requires_manual_review": late_strict or depends_on_reading,
        "policy_refs": ["POL-TIME-01"],
    }


def threshold_check(total_reimbursable):
    """Work out which approval tier a total falls into (POL-APR-01, 02, 03).

    Give this the total after per-diem deductions. That's what the policy says to measure.
    """
    amount = round(float(total_reimbursable), 2)

    if amount <= TIER_AUTO_APPROVE:
        tier = "auto_approve"
        policy_ref = "POL-APR-01"
        within_authority = True
        note = ("$" + format(amount, ",.2f") + " is inside the $"
                + format(TIER_AUTO_APPROVE, ",.0f") + " auto-approval tier.")

    elif amount <= TIER_MANAGER:
        tier = "manager"
        policy_ref = "POL-APR-02"
        within_authority = True
        note = ("$" + format(amount, ",.2f") + " is in the manager tier, which POL-APR-02 treats "
                "as approvable when the claim is otherwise clean.")

    else:
        tier = "director"
        policy_ref = "POL-APR-03"
        within_authority = False
        note = ("$" + format(amount, ",.2f") + " is over $" + format(TIER_MANAGER, ",.0f")
                + ", past what this agent may approve. POL-APR-03 needs a director.")

    return {
        "tool": "threshold_check",
        "total_reimbursable": amount,
        "tier": tier,
        "within_agent_authority": within_authority,
        "requires_manual_review": not within_authority,
        "note": note,
        "policy_refs": [policy_ref],
    }


def duplicate_check(claim_id):
    """Look for the same expense claimed twice across the claim set.

    Counts as a duplicate when the same employee claims the same category and amount over
    overlapping dates. There aren't any in the five sample claims, so this comes back clean on all
    of them. It's here because an expense system with no double-payment check isn't finished, not
    because it catches anything in this sample.
    """
    claim = find_claim(claim_id)
    if claim is None:
        return unknown_claim_error("duplicate_check", claim_id)

    matches = []

    for other in CLAIM_REGISTRY.values():
        if other.claim_id == claim.claim_id:
            continue
        if other.employee != claim.employee:
            continue

        dates_overlap = claim.trip_start <= other.trip_end and other.trip_start <= claim.trip_end
        if not dates_overlap:
            continue

        for item in claim.line_items:
            for other_item in other.line_items:
                same_category = item.category == other_item.category
                same_amount = abs(item.amount - other_item.amount) < 0.01
                if same_category and same_amount:
                    matches.append({
                        "item_id": item.item_id,
                        "duplicate_of": other_item.item_id,
                        "other_claim_id": other.claim_id,
                        "category": item.category,
                        "amount": round(item.amount, 2),
                    })

    return {
        "tool": "duplicate_check",
        "claim_id": claim_id,
        "claims_compared": len(CLAIM_REGISTRY) - 1,
        "duplicates_found": matches,
        "requires_manual_review": len(matches) > 0,
        "policy_refs": [],
    }

## 5. Handing the tools to LangGraph

The functions above are ordinary Python. To let the model call them, each one needs a JSON
schema describing its name, arguments and purpose.

I wrote those schemas out by hand first, which was a mistake: about 150 lines of JSON that
duplicated information already sitting in the function signatures. LangChain's `tool()` builds
the schema from the function itself, taking the name, the parameters and the first line of the
docstring. So the docstring isn't just documentation any more, it's the description the model
reads when it's deciding whether a tool is worth calling. Worth writing carefully.

In [ ]:
from langchain_core.tools import tool

# Wrap each plain function so LangGraph can call it. The schema comes from the function.
TOOLS = [
    tool(policy_lookup),
    tool(eligibility_check),
    tool(per_diem_check),
    tool(receipt_check),
    tool(timeliness_check),
    tool(threshold_check),
    tool(duplicate_check),
]

# Name -> tool, so we can call one directly when we're not going through the agent.
TOOLS_BY_NAME = {}
for wrapped in TOOLS:
    TOOLS_BY_NAME[wrapped.name] = wrapped

print("Tools available to the agent:\n")
for wrapped in TOOLS:
    print("  " + wrapped.name)
    print("      args: " + str(list(wrapped.args.keys())))
    print("      desc: " + wrapped.description.split("\n")[0])

## 6. Unit tests

These asserts are the safety net for the whole assignment. Every number in them I worked out by
hand from the policy before writing any code, so if a later change breaks the lodging cap the
notebook stops here instead of quietly paying out the wrong amount.

The interesting cases are CLM-003 (a $100 lodging deduction, and meals at $70/day that must
*not* be deducted because the cap is $75) and CLM-004 (where the nights conflict holds the
deduction back).

In [ ]:
def test_eligibility():
    result = eligibility_check("CLM-001")
    assert result["eligible_total"] == 1110.00
    assert result["ineligible_total"] == 0.0
    assert result["policy_exceptions"] == []

    result = eligibility_check("CLM-002")
    assert result["all_items_ineligible"] is True, "spa and minibar are both POL-CAT-02"
    assert result["ineligible_total"] == 380.00
    assert result["eligible_total"] == 0.0

    result = eligibility_check("CLM-004")
    assert len(result["policy_exceptions"]) == 1, "business-class airfare is a POL-AIR-01 exception"
    assert result["policy_exceptions"][0]["policy_ref"] == "POL-AIR-01"
    assert result["ineligible_total"] == 0.0, "POL-AIR-01 doesn't make the fare ineligible"


def test_per_diem():
    result = per_diem_check("CLM-001")
    assert result["total_excess"] == 0.0, "2 nights at $180 and 3 days at $60 are both under cap"

    result = per_diem_check("CLM-003")
    lodging = None
    meals = None
    for line in result["lines"]:
        if line["category"] == "lodging":
            lodging = line
        if line["category"] == "meals":
            meals = line
    assert lodging["allowed"] == 400.00, "2 nights x $200 cap"
    assert lodging["excess"] == 100.00
    assert meals["excess"] == 0.0, "$70/day is under the $75 cap, so nothing is deducted"
    assert result["total_excess"] == 100.00

    result = per_diem_check("CLM-004")
    assert result["requires_manual_review"] is True
    conflict = result["conflicts"][0]
    assert conflict["type"] == "unit_count_conflict"
    assert conflict["material"] is True
    assert conflict["excess_if_trip_dates"] == 200.00
    assert conflict["excess_if_description"] == 0.0
    assert result["total_excess"] == 0.0, "a real conflict holds the deduction back"

    result = per_diem_check("CLM-005")
    meals = result["lines"][0]
    assert meals["units_used"] == 1
    assert meals["allowed"] == 75.00
    assert meals["excess"] == 145.00


def test_receipts():
    for claim_id in ["CLM-001", "CLM-002", "CLM-003"]:
        assert receipt_check(claim_id)["missing_docs"] == []

    result = receipt_check("CLM-004")
    assert len(result["missing_docs"]) == 1
    assert "lodging" in result["missing_docs"][0]
    assert result["requires_manual_review"] is True

    result = receipt_check("CLM-005")
    assert len(result["missing_docs"]) == 1, "a $220 meal is over the $25 receipt threshold"


def test_timeliness():
    for claim_id in EXPECTED_TOTALS:
        result = timeliness_check(claim_id)
        assert result["is_late"] is False, claim_id + " was submitted inside the 30-day window"
        assert result["ambiguous_window"] is False


def test_thresholds():
    assert threshold_check(420.00)["tier"] == "auto_approve"
    assert threshold_check(500.00)["tier"] == "auto_approve", "the boundary is inclusive"
    assert threshold_check(500.01)["tier"] == "manager"
    assert threshold_check(840.00)["tier"] == "manager"
    assert threshold_check(2000.00)["tier"] == "manager", "the boundary is inclusive"
    assert threshold_check(2000.01)["tier"] == "director"
    assert threshold_check(3000.00)["within_agent_authority"] is False


def test_duplicates_and_errors():
    for claim_id in EXPECTED_TOTALS:
        assert duplicate_check(claim_id)["duplicates_found"] == []

    assert "error" in eligibility_check("CLM-999"), "unknown ids come back as data, not exceptions"


def test_retrieval():
    assert policy_lookup("POL-PD-02")["policy_refs"] == ["POL-PD-02"], "an explicit id wins"
    assert "POL-PD-02" in policy_lookup("hotel nightly limit")["policy_refs"]


for test in [test_eligibility, test_per_diem, test_receipts, test_timeliness,
             test_thresholds, test_duplicates_and_errors, test_retrieval]:
    test()
    print("passed: " + test.__name__)

print("\nAll tool tests passed.")

In [ ]:
print("Worked example, CLM-003 per-diem. This is the claim that becomes a partial approval:\n")
for line in per_diem_check("CLM-003")["lines"]:
    if line["capped"]:
        print("  " + line["category"].ljust(16)
              + "claimed $" + format(line["amount"], "8,.2f")
              + "   cap $" + format(line["cap_per_unit"], "6,.2f") + "/" + line["unit"]
              + " x " + str(line["units_used"])
              + "   allowed $" + format(line["allowed"], "8,.2f")
              + "   excess $" + format(line["excess"], "7,.2f"))

print("\nWorked example, CLM-004. This is the conflicting-information case:\n")
for conflict in per_diem_check("CLM-004")["conflicts"]:
    print(textwrap.fill(conflict["detail"], width=96, initial_indent="  ", subsequent_indent="  "))

## 7. The rule engine

Before wiring up the agent, here's the same job done with no LLM at all: run every tool, collect
what they say, and apply the decision guidance from the policy.

This exists for three reasons. It's the fallback when there's no API key. It's the yardstick the
guardrail measures the agent's answer against later. And writing it first meant I had the five
correct answers in hand before the model ever got a look at the problem, which is the only way I
could tell whether the agent was right or merely convincing.

The decision order matters and follows the policy's own guidance. Anything that needs a human
wins first. Then a claim with nothing reimbursable is a reject. Then a claim with money over the
caps is a partial. Everything left is an approve.

In [ ]:
def collect_checks(claim_id):
    """Run every check on a claim and return the raw results, keyed by tool name."""
    claim = find_claim(claim_id)
    reimbursable_before_tier = 0.0

    checks = {
        "eligibility_check": eligibility_check(claim_id),
        "per_diem_check": per_diem_check(claim_id),
        "receipt_check": receipt_check(claim_id),
        "timeliness_check": timeliness_check(claim_id),
        "duplicate_check": duplicate_check(claim_id),
    }

    # The tier is judged on what's left after ineligible items and per-diem excess come off.
    eligible = checks["eligibility_check"]["eligible_total"]
    excess = checks["per_diem_check"]["total_excess"]
    reimbursable_before_tier = round(eligible - excess, 2)
    checks["threshold_check"] = threshold_check(reimbursable_before_tier)

    return checks, reimbursable_before_tier


def collect_review_reasons(checks):
    """Gather every reason this claim should go to a human, in plain language."""
    reasons = []

    eligibility = checks["eligibility_check"]
    for exception in eligibility["policy_exceptions"]:
        reasons.append(exception["detail"])
    if eligibility["unrecognised_total"] > 0:
        reasons.append("The claim contains a category the policy does not name either way.")

    for conflict in checks["per_diem_check"]["conflicts"]:
        reasons.append(conflict["detail"])

    receipts = checks["receipt_check"]
    if len(receipts["missing_docs"]) > 0:
        reasons.append(
            "POL-RCT-02: a required receipt is missing, so the claim goes to a reviewer who can "
            "ask for it rather than being rejected outright."
        )

    timeliness = checks["timeliness_check"]
    if timeliness["is_late"]:
        reasons.append("POL-TIME-01: submitted outside the 30-day window.")
    elif timeliness["ambiguous_window"]:
        reasons.append("POL-TIME-01: whether this is late depends on which expense date you use.")

    if len(checks["duplicate_check"]["duplicates_found"]) > 0:
        reasons.append("A line on this claim looks like a duplicate of another claim.")

    threshold = checks["threshold_check"]
    if not threshold["within_agent_authority"]:
        reasons.append(threshold["note"])

    return reasons


def collect_policy_refs(checks):
    """Every rule the checks actually consulted, deduplicated and sorted."""
    refs = set()
    for result in checks.values():
        for ref in result["policy_refs"]:
            refs.add(ref)
    return sorted(refs)


def score_confidence(reasons, checks):
    """How much to trust this recommendation.

    Starts high and comes down for each thing that needed a judgement call. A claim held up by a
    clear rule (a missing receipt, say) stays fairly confident, because the rule is unambiguous.
    A claim where the data contradicts itself gets capped much lower, because we genuinely don't
    know what the right answer is.
    """
    confidence = 0.95
    confidence -= 0.05 * len(reasons)

    has_data_conflict = len(checks["per_diem_check"]["conflicts"]) > 0
    if has_data_conflict and confidence > 0.65:
        confidence = 0.65

    if confidence < 0.40:
        confidence = 0.40

    return round(confidence, 2)


def decide_deterministically(claim_id):
    """Work out the decision and the amounts using only the tools. No LLM anywhere in here."""
    checks, reimbursable = collect_checks(claim_id)
    reasons = collect_review_reasons(checks)

    eligibility = checks["eligibility_check"]
    ineligible = eligibility["ineligible_total"]
    excess = checks["per_diem_check"]["total_excess"]

    if len(reasons) > 0:
        # Something needs a person. Don't award any money yet.
        decision = Decision.MANUAL_REVIEW
        approved = 0.0
        deducted = 0.0

    elif eligibility["all_items_ineligible"]:
        decision = Decision.REJECT
        approved = 0.0
        deducted = round(ineligible, 2)

    elif excess > 0 or ineligible > 0:
        decision = Decision.PARTIAL_APPROVE
        approved = reimbursable
        deducted = round(excess + ineligible, 2)

    else:
        decision = Decision.APPROVE
        approved = reimbursable
        deducted = 0.0

    return {
        "claim_id": claim_id,
        "decision": decision,
        "approved_amount": approved,
        "deducted_amount": deducted,
        "missing_docs": checks["receipt_check"]["missing_docs"],
        "policy_refs": collect_policy_refs(checks),
        "confidence": score_confidence(reasons, checks),
        "review_reasons": reasons,
        "checks": checks,
        "reimbursable_if_approved": reimbursable,
    }

In [ ]:
# The answer key. I worked these out by hand from the policy before writing the code above.
EXPECTED_DECISIONS = {
    "CLM-001": (Decision.APPROVE, 1110.00, 0.00),
    "CLM-002": (Decision.REJECT, 0.00, 380.00),
    "CLM-003": (Decision.PARTIAL_APPROVE, 840.00, 100.00),
    "CLM-004": (Decision.MANUAL_REVIEW, 0.00, 0.00),
    "CLM-005": (Decision.MANUAL_REVIEW, 0.00, 0.00),
}

print("Rule engine results:\n")
print("claim     decision          approved   deducted   conf   why")
print("-" * 100)

for claim_id in sorted(EXPECTED_DECISIONS):
    result = decide_deterministically(claim_id)

    expected_decision, expected_approved, expected_deducted = EXPECTED_DECISIONS[claim_id]
    assert result["decision"] == expected_decision, (
        claim_id + ": got " + result["decision"].value + ", expected " + expected_decision.value
    )
    assert result["approved_amount"] == expected_approved, claim_id + " approved amount"
    assert result["deducted_amount"] == expected_deducted, claim_id + " deducted amount"

    if len(result["review_reasons"]) > 0:
        why = str(len(result["review_reasons"])) + " review trigger(s)"
    else:
        why = "clean"

    print(claim_id + "   "
          + result["decision"].value.ljust(16)
          + format(result["approved_amount"], ">9,.2f") + "  "
          + format(result["deducted_amount"], ">9,.2f") + "   "
          + format(result["confidence"], ".2f") + "   "
          + why)

print("\nAll five match the expected decisions.")

In [ ]:
# Why CLM-004 needs a person: four separate triggers, any one of which would be enough.
print("CLM-004 review triggers:\n")
for number, reason in enumerate(decide_deterministically("CLM-004")["review_reasons"], start=1):
    print(textwrap.fill(str(number) + ". " + reason, width=96, subsequent_indent="   "))
    print()

## 8. The agent

Now the agentic part. The rule engine above follows a fixed order that I chose when I wrote it.
The agent doesn't: it gets the claim and the tools, and works out for itself what to check and
in what order.

The graph has two nodes and one decision:

```
        ┌──────────────┐
  START │              │
    └──▶│    agent     │  the model reads the conversation and either
        │              │  asks for a tool or gives its answer
        └──┬────────┬──┘
           │        │
    asked  │        │  no tool call
  for a    │        │  ▼
  tool     │       END
           ▼
        ┌──────────────┐
        │    tools     │  ToolNode runs whatever it asked for
        └──────┬───────┘  and appends the results
               │
               └────────▶ back to agent
```

That loop is the whole idea. The model calls `eligibility_check`, sees a business-class fare in
the reply, and *because of what it saw* decides to call `policy_lookup` for the airfare rule.
Nobody scripted that sequence. A fixed pipeline can't do it.

`MAX_AGENT_TURNS` caps the loop. A model that gets confused and keeps calling the same tool would
otherwise run until the money ran out.

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

SYSTEM_PROMPT = """You review employee travel reimbursement claims against company policy.

Work through the claim using the tools. A reasonable order is: check eligibility, check per-diem
caps, check receipts, check timeliness, check for duplicates, then check the approval tier on
whatever is left. Use policy_lookup whenever you need the exact wording of a rule.

Rules you must follow:

1. Never calculate an amount yourself. Every figure you report must come from a tool result.
   If you need the approval tier, call threshold_check with the total after deductions.
2. Only cite POL-* ids that appeared in a tool result.
3. If a tool sets requires_manual_review, the claim goes to MANUAL_REVIEW. Do not talk yourself
   out of it.
4. When information is missing or two pieces of the claim contradict each other, choose
   MANUAL_REVIEW. Never guess an amount to avoid an awkward answer.

How the decisions work:

{guidance}

When you have run the checks you need, stop calling tools and write a short summary of what you
found and what should happen to the claim."""


def build_model():
    """Create the chat model, with the tools attached."""
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(model=MODEL, temperature=LLM_TEMPERATURE, api_key=API_KEY)
    else:
        from langchain_groq import ChatGroq
        model = ChatGroq(model=MODEL, temperature=LLM_TEMPERATURE, api_key=API_KEY)

    return model.bind_tools(TOOLS)


def build_graph(model_with_tools):
    """Wire the agent node and the tool node into a loop."""

    def call_model(state):
        """Ask the model what to do next, given everything said so far."""
        reply = model_with_tools.invoke(state["messages"])
        return {"messages": [reply]}

    def should_continue(state):
        """If the model asked for a tool, run it. Otherwise we're done."""
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
            return "tools"
        return END

    graph = StateGraph(MessagesState)
    graph.add_node("agent", call_model)
    graph.add_node("tools", ToolNode(TOOLS))

    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, ["tools", END])
    graph.add_edge("tools", "agent")

    return graph.compile()


if LLM_AVAILABLE:
    agent_graph = build_graph(build_model())
    print("Agent graph compiled.\n")
    print(agent_graph.get_graph().draw_mermaid())
else:
    agent_graph = None
    print("No API key, so the agent graph is not built. The rule engine handles everything.")

### Getting a structured answer out

When the loop finishes, the agent has written a paragraph. We need nine typed fields.

I could ask it to reply in JSON and parse that, but then I'd be handling the case where it wraps
the JSON in a code fence, or writes a trailing comma, or explains itself first. `with_structured_output`
avoids all of it: hand it the Pydantic model and the reply comes back as a validated object,
retried by the library if the model gets the shape wrong.

Notice what the agent is *not* asked for: `tools_used`. That gets read off the message history,
so it records what actually ran rather than what the model remembers running.

In [ ]:
class AgentVerdict(BaseModel):
    """What we ask the model for at the end. Deliberately smaller than the final output."""

    decision: Decision
    approved_amount: float = Field(ge=0, description="Amount to reimburse, from the tool results")
    deducted_amount: float = Field(ge=0, description="Amount refused, from the tool results")
    missing_docs: list[str] = Field(default_factory=list)
    policy_refs: list[str] = Field(default_factory=list, description="Only POL-* ids seen in tool results")
    confidence: float = Field(ge=0, le=1, description="Lower this when the claim is ambiguous")
    explanation: str = Field(description="Two to four sentences a claimant would understand")


VERDICT_PROMPT = """Summarise your review as structured fields.

Use only amounts that appeared in tool results. If you decided MANUAL_REVIEW, set approved_amount
and deducted_amount to 0, because nothing has been decided yet."""


def tools_actually_called(messages):
    """Read the tool names out of the message history, in the order they ran."""
    called = []
    for message in messages:
        tool_calls = getattr(message, "tool_calls", None)
        if not tool_calls:
            continue
        for call in tool_calls:
            name = call["name"]
            if name not in called:
                called.append(name)
    return called


def build_audit_trail(messages):
    """A readable log of what the agent did, for the UI and for debugging."""
    trail = []
    for message in messages:
        kind = message.__class__.__name__

        if kind == "AIMessage":
            tool_calls = getattr(message, "tool_calls", None)
            if tool_calls:
                for call in tool_calls:
                    trail.append({
                        "step": "calls tool",
                        "detail": call["name"] + "(" + json.dumps(call["args"]) + ")",
                    })
            elif message.content:
                trail.append({"step": "concludes", "detail": message.content})

        elif kind == "ToolMessage":
            summary = message.content
            if len(summary) > 220:
                summary = summary[:220] + " ..."
            trail.append({"step": "tool returns", "detail": message.name + " -> " + summary})

    return trail


def run_agent(claim_id):
    """Run the agent loop on one claim, then ask it for a structured verdict."""
    claim = find_claim(claim_id)

    system_prompt = SYSTEM_PROMPT.format(guidance=DECISION_GUIDANCE)
    opening_message = (
        "Review this claim and recommend what should happen to it.\n\n"
        + claim.model_dump_json(indent=2)
    )

    conversation = agent_graph.invoke(
        {"messages": [("system", system_prompt), ("human", opening_message)]},
        {"recursion_limit": MAX_AGENT_TURNS * 2},
    )
    messages = conversation["messages"]

    # Second call: same conversation, but now asking for typed fields instead of prose.
    verdict_model = build_model().with_structured_output(AgentVerdict)
    verdict = verdict_model.invoke(messages + [("human", VERDICT_PROMPT)])

    return {
        "verdict": verdict,
        "tools_used": tools_actually_called(messages),
        "audit_trail": build_audit_trail(messages),
        "turns": len(messages),
    }

## 9. Testing the agent loop without an API

I wanted to prove the graph works before spending anything, and a reviewer without a key should
be able to see it work too. So this swaps the real model for one that replays a fixed list of
replies. Everything else is real: the same graph, the same `ToolNode`, the same tools.

What it proves is the loop actually loops. The scripted model asks for one tool, gets a real
result appended to the conversation, asks for another, then stops. Six messages come out of a
two-node graph, which only happens if control really did go agent, tools, agent, tools, agent.

It also checks the two things I don't want to take the model's word for: that `tools_used` is read
from the message history, and that the audit trail lines up with what ran.

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult


class ScriptedModel(BaseChatModel):
    """Stands in for a real model by replaying a fixed list of replies.

    Once the list runs out it keeps returning the last reply, which is how we test that the turn cap
    stops a model that will not stop calling tools.

    One trap worth knowing about. LangGraph's message reducer treats two messages with the same id
    as the same message and replaces rather than appends. Handing back the same AIMessage object
    twice therefore wipes out the earlier turn and the loop quietly ends. So every reply gets built
    fresh here, with its own id.
    """

    replies: list = []
    call_count: list = []

    @property
    def _llm_type(self):
        return "scripted"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        index = min(len(self.call_count), len(self.replies) - 1)
        self.call_count.append(index)
        template = self.replies[index]
        turn = str(len(self.call_count))

        fresh_tool_calls = []
        for call in template.tool_calls or []:
            fresh_tool_calls.append({
                "name": call["name"],
                "args": dict(call["args"]),
                "id": call["id"] + "_turn" + turn,
            })

        reply = AIMessage(
            content=template.content,
            tool_calls=fresh_tool_calls,
            id="scripted_" + turn,
        )
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def bind_tools(self, tools, **kwargs):
        return self


def ask_for_tool(name, claim_id, call_id):
    """Build the message a model sends when it wants a tool run."""
    return AIMessage(
        content="",
        tool_calls=[{"name": name, "args": {"claim_id": claim_id}, "id": call_id}],
    )


def test_agent_loop():
    """The graph should run each requested tool and come back for more until the model stops."""
    script = [
        ask_for_tool("eligibility_check", "CLM-003", "call_1"),
        ask_for_tool("per_diem_check", "CLM-003", "call_2"),
        AIMessage(content="Lodging is $100 over the nightly cap. This is a partial approval."),
    ]
    graph = build_graph(ScriptedModel(replies=script, call_count=[]))
    conversation = graph.invoke({"messages": [("human", "Review CLM-003")]})
    messages = conversation["messages"]

    # human, ai, tool, ai, tool, ai
    assert len(messages) == 6, "expected 6 messages, got " + str(len(messages))

    # The tool results in the conversation are real output from our real tools.
    per_diem_reply = json.loads(messages[4].content)
    assert per_diem_reply["total_excess"] == 100.00

    assert tools_actually_called(messages) == ["eligibility_check", "per_diem_check"]
    assert len(build_audit_trail(messages)) == 5


def test_turn_cap_stops_a_stuck_model():
    """A model that only ever asks for tools must be stopped, not left to run."""
    stuck = [ask_for_tool("eligibility_check", "CLM-001", "call_1")]
    graph = build_graph(ScriptedModel(replies=stuck, call_count=[]))

    hit_the_limit = False
    try:
        graph.invoke(
            {"messages": [("human", "Review CLM-001")]},
            {"recursion_limit": MAX_AGENT_TURNS},
        )
    except Exception as error:
        # LangGraph raises GraphRecursionError once the limit is reached.
        hit_the_limit = "recursion" in str(error).lower() or "limit" in str(error).lower()

    assert hit_the_limit, "a model stuck in a tool-calling loop should hit the turn cap"


def test_agent_can_be_told_about_a_bad_claim_id():
    """An unknown claim id should come back as a readable message, not blow up the graph."""
    script = [
        ask_for_tool("eligibility_check", "CLM-999", "call_1"),
        AIMessage(content="That claim id does not exist."),
    ]
    graph = build_graph(ScriptedModel(replies=script, call_count=[]))
    conversation = graph.invoke({"messages": [("human", "Review CLM-999")]})

    tool_reply = json.loads(conversation["messages"][2].content)
    assert "error" in tool_reply
    assert "CLM-999" in tool_reply["error"]


for test in [test_agent_loop, test_turn_cap_stops_a_stuck_model,
             test_agent_can_be_told_about_a_bad_claim_id]:
    test()
    print("passed: " + test.__name__)

print("\nThe agent loop works. No API calls were made.")

## 10. The guardrail

The agent can be wrong. It might add up tool results incorrectly, cite a rule it never looked at,
or talk itself into approving something it shouldn't. So nothing it says goes straight into the
output.

The check is simple: the rule engine decides the same claim independently, and the two have to
agree. Same idea as having two people check a payment, or double-entry bookkeeping. Specifically:

- **Amounts always come from the rule engine.** Not because the agent is usually wrong, but
  because the rule engine's answer is reproducible and the agent's isn't.
- **If the two decisions differ, the claim goes to a human.** A disagreement means the claim isn't
  as clear-cut as one of them thought, and that's exactly what manual review is for.
- **Invented policy ids get dropped**, checked against the twelve ids that came out of the policy
  document.
- **Low confidence forces manual review**, under `CONFIDENCE_FLOOR`.

The agent still does real work here. It decides which checks to run and in what order, it's the
reason the decision is cross-checked at all, and the explanation in the output is its writing. It
just doesn't get the last word on money.

One deliberate choice: the nine output fields are fixed by the assignment, so there's nowhere to
record that the guardrail intervened. I keep those notes in a separate structure for the UI and
the audit trail rather than bending the output shape.

In [ ]:
def drop_invented_refs(refs):
    """Keep only rule ids that exist in the policy document."""
    kept = []
    for ref in refs:
        if ref in VALID_POLICY_IDS and ref not in kept:
            kept.append(ref)
    return sorted(kept)


def apply_guardrail(claim_id, verdict, tools_used):
    """Check the agent's verdict against the rule engine and build the final result.

    Returns the validated ClaimResult plus a list of notes about anything we had to correct.
    """
    baseline = decide_deterministically(claim_id)
    notes = []

    # 1. Policy ids the agent cited that aren't in the document.
    invented = []
    for ref in verdict.policy_refs:
        if ref not in VALID_POLICY_IDS:
            invented.append(ref)
    if len(invented) > 0:
        notes.append("Dropped policy ids that are not in the document: " + ", ".join(invented))

    # Cite the union of what the agent said and what the checks consulted, minus anything invented.
    refs = drop_invented_refs(list(verdict.policy_refs) + baseline["policy_refs"])

    # 2. Do the two decisions agree?
    decisions_agree = verdict.decision == baseline["decision"]
    if not decisions_agree:
        notes.append(
            "The agent said " + verdict.decision.value + " but the rule engine said "
            + baseline["decision"].value + ". Sending to a human because they disagree."
        )

    # 3. Did the agent's amounts match what the tools produced?
    if verdict.decision == baseline["decision"]:
        approved_gap = abs(verdict.approved_amount - baseline["approved_amount"])
        deducted_gap = abs(verdict.deducted_amount - baseline["deducted_amount"])
        if approved_gap >= 0.01 or deducted_gap >= 0.01:
            notes.append(
                "The agent's amounts (approved $" + format(verdict.approved_amount, ",.2f")
                + ", deducted $" + format(verdict.deducted_amount, ",.2f") + ") did not match the "
                "tool results. Used the tool figures instead."
            )

    # 4. Confidence. Take the lower of the two, so neither side can talk it up.
    confidence = min(verdict.confidence, baseline["confidence"])
    if confidence < CONFIDENCE_FLOOR:
        notes.append(
            "Confidence " + format(confidence, ".2f") + " is under the "
            + format(CONFIDENCE_FLOOR, ".2f") + " floor, so this goes to a human."
        )

    # Decide the final outcome.
    needs_human = (not decisions_agree) or confidence < CONFIDENCE_FLOOR
    if needs_human:
        decision = Decision.MANUAL_REVIEW
        approved = 0.0
        deducted = 0.0
    else:
        decision = baseline["decision"]
        approved = baseline["approved_amount"]
        deducted = baseline["deducted_amount"]

    explanation = verdict.explanation.strip()
    if len(notes) > 0:
        explanation = explanation + " (Automated check: " + " ".join(notes) + ")"

    result = ClaimResult(
        claim_id=claim_id,
        decision=decision,
        approved_amount=approved,
        deducted_amount=deducted,
        missing_docs=baseline["missing_docs"],
        policy_refs=refs,
        confidence=confidence,
        explanation=explanation,
        tools_used=tools_used,
    )
    return result, notes

In [ ]:
def test_guardrail_accepts_a_correct_verdict():
    """When the agent agrees with the rule engine, its verdict stands."""
    honest = AgentVerdict(
        decision=Decision.PARTIAL_APPROVE,
        approved_amount=840.00,
        deducted_amount=100.00,
        policy_refs=["POL-PD-02", "POL-APR-02"],
        confidence=0.9,
        explanation="Lodging exceeded the nightly cap, so $100 was deducted.",
    )
    result, notes = apply_guardrail("CLM-003", honest, ["per_diem_check"])

    assert result.decision == Decision.PARTIAL_APPROVE
    assert result.approved_amount == 840.00
    assert notes == [], "a correct verdict needs no corrections"


def test_guardrail_catches_wrong_arithmetic():
    """An agent that gets the sums wrong should not be able to set the payout."""
    bad_maths = AgentVerdict(
        decision=Decision.PARTIAL_APPROVE,
        approved_amount=940.00,   # forgot to take the $100 off
        deducted_amount=0.00,
        policy_refs=["POL-PD-02"],
        confidence=0.9,
        explanation="Approving the lodging in full.",
    )
    result, notes = apply_guardrail("CLM-003", bad_maths, ["per_diem_check"])

    assert result.approved_amount == 840.00, "the tool figure wins"
    assert result.deducted_amount == 100.00
    assert len(notes) == 1
    assert "did not match" in notes[0]


def test_guardrail_blocks_an_over_generous_approval():
    """An agent that approves a claim needing a human gets overruled."""
    too_keen = AgentVerdict(
        decision=Decision.APPROVE,
        approved_amount=3000.00,
        deducted_amount=0.00,
        policy_refs=["POL-CAT-01"],
        confidence=0.95,
        explanation="All looks fine to me.",
    )
    result, notes = apply_guardrail("CLM-004", too_keen, ["eligibility_check"])

    assert result.decision == Decision.MANUAL_REVIEW
    assert result.approved_amount == 0.00, "nothing is awarded when a human still has to look"
    assert "disagree" in notes[0]


def test_guardrail_drops_invented_policy_ids():
    """A rule id that isn't in the policy must not reach the output."""
    makes_things_up = AgentVerdict(
        decision=Decision.REJECT,
        approved_amount=0.00,
        deducted_amount=380.00,
        policy_refs=["POL-CAT-02", "POL-SPA-99"],
        confidence=0.9,
        explanation="Spa and minibar are not reimbursable.",
    )
    result, notes = apply_guardrail("CLM-002", makes_things_up, ["eligibility_check"])

    assert "POL-SPA-99" not in result.policy_refs
    assert "POL-CAT-02" in result.policy_refs
    assert "POL-SPA-99" in notes[0]


def test_guardrail_respects_the_confidence_floor():
    """Even when both sides agree, an unsure verdict goes to a human."""
    unsure = AgentVerdict(
        decision=Decision.APPROVE,
        approved_amount=1110.00,
        deducted_amount=0.00,
        policy_refs=["POL-APR-02"],
        confidence=0.4,
        explanation="I think this is fine but the purpose is vague.",
    )
    result, notes = apply_guardrail("CLM-001", unsure, ["eligibility_check"])

    assert result.decision == Decision.MANUAL_REVIEW
    assert "floor" in notes[-1]


for test in [test_guardrail_accepts_a_correct_verdict,
             test_guardrail_catches_wrong_arithmetic,
             test_guardrail_blocks_an_over_generous_approval,
             test_guardrail_drops_invented_policy_ids,
             test_guardrail_respects_the_confidence_floor]:
    test()
    print("passed: " + test.__name__)

print("\nThe guardrail holds. An agent cannot overpay, invent a rule, or approve past its authority.")